In [ ]:
import os

for dirname, dirnames, filenames in os.walk('/kaggle/input/'):
    # Print the current folder
    print(f'Folder: {dirname}')
    
    # Print all files in this folder with an indentation
    for filename in filenames:
        print(f'    File: {filename}')

In [1]:
!git clone https://github.com/NguyenPhuHung2006/AI_Training.git

Cloning into 'AI_Training'...
remote: Enumerating objects: 2005, done.
remote: Counting objects: 100% (867/867), done.
remote: Compressing objects: 100% (482/482), done.
remote: Total 2005 (delta 447), reused 769 (delta 358), pack-reused 1138 (from 1)
Receiving objects: 100% (2005/2005), 149.87 MiB | 31.67 MiB/s, done.
Resolving deltas: 100% (638/638), done.


In [2]:
import os
import sys
sys.path.append('/kaggle/working/AI_Training/ai_model')
import pandas as pd
import torch
import os
from torch.nn.utils.rnn import pad_sequence
from ai_model.deep_learning.nn_torch import RNN
from ai_model.deep_learning.nn_torch.callback import EarlyStopping, ModelCheckpoint
from tokenizers import ByteLevelBPETokenizer, Tokenizer
import numpy as np

In [3]:
import pandas as pd
import torch
import os
import numpy as np
from torch.nn.utils.rnn import pad_sequence
from ai_model.deep_learning.nn_torch import RNN
from ai_model.deep_learning.nn_torch.callback import EarlyStopping, ModelCheckpoint
from tokenizers import Tokenizer

def preprocessing_data(file_path, has_label=True):
    target_cols = ["ID", "code", "Label"] if has_label else ["ID", "code"]
    
    # 1. Load data with initial type hints to save memory
    df = pd.read_csv(file_path, usecols=target_cols, low_memory=False)
    
    # 2. Clean 'code' column (must not be empty)
    df["code"] = df["code"].fillna("").astype(str)
    df = df[df["code"].str.strip() != ""]
    
    # 3. Handle 'ID' column (force to numeric, drop non-numeric)
    df["ID"] = pd.to_numeric(df["ID"], errors='coerce')
    df = df.dropna(subset=["ID"])
    df["ID"] = df["ID"].astype(int)
    
    if has_label:
        # 4. Handle 'Label' column
        # errors='coerce' turns strings/garbage into NaN
        df["Label"] = pd.to_numeric(df["Label"], errors='coerce')
        
        # 5. Drop rows with missing labels
        df = df.dropna(subset=["Label"])
        
        # 6. Strictly keep only binary 0 and 1
        # This prevents the negative loss issue
        df = df[df["Label"].isin([0, 1])]
        df["Label"] = df["Label"].astype(int)
        
    return df

def get_tokenize(df, tokenizer, MAX_T=512, has_label=True, pad_id=0):
    encoded_codes = []
    for text in df["code"]:
        token_ids = tokenizer.encode(text).ids
        # Strategy: Take the first MAX_T tokens
        # If the code is longer than MAX_T, we truncate. 
        # For code, the "intent" is often at the top.
        truncated_ids = token_ids[:MAX_T]
        encoded_codes.append(torch.tensor(truncated_ids, dtype=torch.long))
    
    # This creates a (Batch, Max_Len_in_Batch) tensor
    padded_sequences = pad_sequence(encoded_codes, batch_first=True, padding_value=pad_id)
    
    labels_tensor = None
    if has_label:
        labels_tensor = torch.tensor(df["Label"].tolist(), dtype=torch.float).unsqueeze(1)
    
    return padded_sequences, labels_tensor

def main():
    # Paths
    DATA_DIR = "/kaggle/input/datasets/nguyenphuhung2006/code-data"
    tok_path = f"{DATA_DIR}/tokenizer.json"
    embed_path = f"{DATA_DIR}/embedding.pt"
    
    # 1. Load Data
    print("Loading data...")
    df_train = preprocessing_data(f"{DATA_DIR}/train_clean_code.csv")
    df_test = preprocessing_data(f"{DATA_DIR}/test_clean_code.csv", has_label=False)

    print(df_train["Label"].unique())
    
    # 2. Tokenization
    tokenizer = Tokenizer.from_file(tok_path)
    pad_id = tokenizer.token_to_id("<pad>")
    
    # Set MAX_T high enough to capture logic but low enough for memory
    # 512 is a good balance for Kaggle P100/T4 GPUs
    MAX_T = 512
    
    print(f"Tokenizing sequences (Max Length: {MAX_T})...")
    X_train, y_train = get_tokenize(df_train, tokenizer, MAX_T=MAX_T, pad_id=pad_id)
    X_test, _ = get_tokenize(df_test, tokenizer, MAX_T=MAX_T, pad_id=pad_id, has_label=False)

    # 3. Model Setup
    model = RNN(
        mode="many_to_one", 
        cost="bce", 
        lr=1e-4,          # Lower learning rate for longer sequences
        use_packing=True, 
        pad_id=pad_id,
        weight_decay=1e-4 # Added weight decay to fight overfitting
    )
    
    # Load and set pre-trained embeddings
    embedding_matrix = torch.load(embed_path, weights_only=False)
    model.set_embedding_matrix(embedding_matrix, freeze=False)
    
    # Model Architecture
    model.add_rnn(hidden_size=128, num_layers=2, dropout=0.4, bidirectional=True)
    model.add_attention()
    model.add_fc(64, activation="relu", dropout=0.3) # Intermediate layer
    model.add_fc(1)
    
    model.build()
    
    # Learning rate scheduler - will reduce LR if val_loss stops improving
    model.set_scheduler("reduce_on_plateau", factor=0.5, patience=2)
    
    # 4. Training
    os.makedirs("nn_data", exist_ok=True)
    callbacks = [
        EarlyStopping(patience=5), 
        ModelCheckpoint("nn_data/best_rnn.pth")
    ]
    
    print("Starting Training...")
    model.fit(
        X_train, 
        y_train, 
        epochs=100, 
        batch_size=32,   # Smaller batch size for long sequences
        val_split=0.15, 
        callbacks=callbacks
    )
    
    # 5. Prediction & Submission
    print("Generating predictions...")
    model.load("nn_data/best_rnn.pth")
    y_pred = model.predict(X_test)
    y_pred = (y_pred > 0.0).astype(int).flatten()
    
    # Ensure ID range 0-6999
    df_sub = pd.DataFrame({"ID": df_test["ID"].tolist(), "Label": y_pred})
    df_final = pd.DataFrame({"ID": np.arange(0, 7000)})
    df_final = df_final.merge(df_sub, on="ID", how="left").fillna(0)
    df_final["Label"] = df_final["Label"].astype(int)
    
    os.makedirs("outputs", exist_ok=True)
    df_final.to_csv("outputs/submission.csv", index=False)
    print("Completed! File saved to outputs/submission.csv")

if __name__ == "__main__":
    main()

Loading data...
[0 1]
Tokenizing sequences (Max Length: 512)...
Starting Training...


Training:   1%|          | 1/100 [00:32<53:17, 32.30s/it]

0 | train_loss:0.6901 | train_acc:53.80% | val_loss:0.6900 | val_acc:53.95%


Training:   2%|▏         | 2/100 [01:04<52:37, 32.21s/it]

1 | train_loss:0.6897 | train_acc:54.41% | val_loss:0.6897 | val_acc:53.95%


...

Training:  15%|█▌        | 15/100 [07:53<45:03, 31.80s/it]

14 | train_loss:0.6227 | train_acc:64.65% | val_loss:0.6691 | val_acc:59.38%


Training:  16%|█▌        | 16/100 [08:23<44:05, 31.50s/it]

Early stopping triggered
15 | train_loss:0.6184 | train_acc:64.94% | val_loss:0.6822 | val_acc:58.93%
Generating predictions...


Completed! File saved to outputs/submission.csv


In [7]:
import pandas as pd
import torch
import os
from torch.nn.utils.rnn import pad_sequence
from ai_model.deep_learning.nn_torch import TextCNN
from ai_model.deep_learning.nn_torch.callback import EarlyStopping, ModelCheckpoint
from tokenizers import ByteLevelBPETokenizer, Tokenizer
import numpy as np

def preprocessing_data(file_path, has_label=True):
    target_cols = ["ID", "code", "Label"] if has_label else ["ID", "code"]
    
    # 1. Load data with initial type hints to save memory
    df = pd.read_csv(file_path, usecols=target_cols, low_memory=False)
    
    # 2. Clean 'code' column (must not be empty)
    df["code"] = df["code"].fillna("").astype(str)
    df = df[df["code"].str.strip() != ""]
    
    # 3. Handle 'ID' column (force to numeric, drop non-numeric)
    df["ID"] = pd.to_numeric(df["ID"], errors='coerce')
    df = df.dropna(subset=["ID"])
    df["ID"] = df["ID"].astype(int)
    
    if has_label:
        # 4. Handle 'Label' column
        # errors='coerce' turns strings/garbage into NaN
        df["Label"] = pd.to_numeric(df["Label"], errors='coerce')
        
        # 5. Drop rows with missing labels
        df = df.dropna(subset=["Label"])
        
        # 6. Strictly keep only binary 0 and 1
        # This prevents the negative loss issue
        df = df[df["Label"].isin([0, 1])]
        df["Label"] = df["Label"].astype(int)
        
    return df
        
def tokenizing(df):
    with open("temp_code.txt", "w", encoding="utf-8") as f:
        for code_snippet in df["code"]:
            f.write(code_snippet + "\n")

    tokenizer = ByteLevelBPETokenizer()
    tokenizer.add_special_tokens([
        "<pad>",
        "<sos>",
        "<eos>"
    ])
    tokenizer.train(files=["temp_code.txt"], vocab_size=5000, min_frequency=2)
    os.remove("temp_code.txt")
    
    return tokenizer
    
def get_tokenize(df, tokenizer, MAX_T=1024, has_label=True, pad_id=None):
    encoded_codes = []
    for text in df["code"]:
        token_ids = tokenizer.encode(text).ids
        if len(token_ids) > MAX_T:
            head_size = int(MAX_T * 0.2)
            tail_size = MAX_T - head_size
            truncated_ids = token_ids[:head_size] + token_ids[-tail_size:]
        else:
            truncated_ids = token_ids
        encoded_codes.append(torch.tensor(truncated_ids, dtype=torch.long))

    pad_id = pad_id if pad_id is not None else 0
    padded_sequences = pad_sequence(encoded_codes, batch_first=True, padding_value=pad_id)
    
    labels_tensor = None
    if has_label:
        labels_tensor = torch.tensor(df["Label"].tolist(), dtype=torch.float).unsqueeze(1)
    
    return padded_sequences, labels_tensor

def main():
    DATA_DIR = "/kaggle/input/datasets/nguyenphuhung2006/code-data"
    df_train = preprocessing_data(f"{DATA_DIR}/train_clean_code.csv")
    df_test = preprocessing_data(f"{DATA_DIR}/test_clean_code.csv", has_label=False)

    print(df_train["Label"].unique())
    
    # tokenizer = tokenizing(df_train)
    tokenizer = Tokenizer.from_file(f"{DATA_DIR}/tokenizer.json")
    pad_id = tokenizer.token_to_id("<pad>")
    
    MAX_T = 512
    X_train, y_train = get_tokenize(df_train, tokenizer, has_label=True, MAX_T=MAX_T, pad_id=pad_id)
    X_test, _ = get_tokenize(df_test, tokenizer, has_label=False, MAX_T=MAX_T, pad_id=pad_id)
    vocab_size = tokenizer.get_vocab_size()
    
    model = TextCNN(
        cost="bce",
        lr=3e-4,
        weight_decay=1e-4,
        pad_id=pad_id
    )

    # model.add_embedding(vocab_size=vocab_size, embed_dim=256)
    embedding_matrix = torch.load(f"{DATA_DIR}/embedding.pt", weights_only=False)
    model.set_embedding_matrix(embedding_matrix)
    model.embedding.weight.requires_grad = True

    for k in [3, 5, 7, 9, 11, 15, 21]:
        model.add_filter(out_channels=128, kernel_size=k, activation="relu", dropout=0.3)

    model.add_pool("max", pool_dropout=0.4)

    model.add_fc(n_units=256, activation="relu", dropout=0.4)
    model.add_fc(n_units=1)
    model.build()
    
    model.set_scheduler("reduce_on_plateau", mode="min", factor=0.5, patience=2)
    
    nn_data_path = "nn_data"
    os.makedirs(nn_data_path, exist_ok=True)
    
    i = 0
    while os.path.exists(f"{nn_data_path}/text_cnn_{i}.pth"):
        i += 1
        
    # reload the model
    model.load(f"{nn_data_path}/text_cnn_{i - 1}.pth")

    callbacks = [EarlyStopping(patience=7), ModelCheckpoint(f"{nn_data_path}/text_cnn_{i}.pth")]
    
    print(f"Starting training with Vocab Size: {vocab_size} and Max Sequence: {X_train.shape[1]}")
    
    # model.fit(
    #     X_train, 
    #     y_train, 
    #     epochs=100, 
    #     batch_size=32, 
    #     callbacks=callbacks
    # )
    
    y_pred = model.predict(X_test)
    y_pred = (y_pred > 0.0).astype(int)
    y_pred = y_pred.flatten()
    
    df = pd.DataFrame({
        "ID": df_test["ID"].tolist(),
        "Label": y_pred
    })
    
    df_temp = pd.DataFrame({
        "ID": np.arange(0, 7000)
    })
    
    df_final = df_temp.merge(df, on="ID", how="left")
    df_final["Label"] = df_final["Label"].fillna(0).astype(int)
    
    output_path = "outputs/nn"
    os.makedirs(output_path, exist_ok=True)

    df_final.to_csv(f"{output_path}/text_cnn_final.csv", index=False)
    

if __name__ == "__main__":
    main()
    print("Completed!")

[0 1]
Starting training with Vocab Size: 10003 and Max Sequence: 512
Completed!
